This script will overlay larger grids on the 4km pixels and determine if they are burned or unburned by using a 15% requirement of pixels that need to be burned in order to classify it as burned.  It will save tif files and parquet files. I will also give a unique ID to the lareger 1:10 degree cells which will be saved in the parquet files. 

Now lets actually save the shapefiles etc for this like before

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio as rio
from rasterio.transform import from_origin
from rasterio.features import rasterize
from tqdm import tqdm

import geopandas as gpd
from shapely.geometry import Polygon
from pyproj import Transformer

# ----------------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------------
OUT_DIR = "/explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction"

PARQUET_DIR    = Path(OUT_DIR) / "parquet_coarse_grids_annual_analytical"
COARSE_TIF_DIR = Path(OUT_DIR) / "tifs_coarse_grids_annual_analytical"
COARSE_SHP_DIR = Path(OUT_DIR) / "shp_coarse_grids_annual_analytical"

os.makedirs(PARQUET_DIR, exist_ok=True)
os.makedirs(COARSE_TIF_DIR, exist_ok=True)
os.makedirs(COARSE_SHP_DIR, exist_ok=True)

# ----------------------------------------------------------------------
# CONSTANTS
# ----------------------------------------------------------------------
WANTED = [
    "DEM", "slope", "aspect", "b1", "relative_humidity",
    "total_precipitation_sum", "temperature_2m", "temperature_2m_min",
    "temperature_2m_max", "build_up_index", "drought_code",
    "duff_moisture_code", "fine_fuel_moisture_code", "fire_weather_index",
    "initial_fire_spread_index",
]

GRID_SIZES_DEG      = [1]
BURNED_THRESHOLD    = 0.05
FRACTION_BAND_NAME  = "fraction"
WRITE_QA_LABEL_ON_4KM = False

# ----------------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------------
def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", s.lower())

WANTED_NORM   = [_norm(x) for x in WANTED]
FRACTION_NORM = _norm(FRACTION_BAND_NAME)
name_re = re.compile(r"cems_e5l_firecci_(\d{4})_(\d{1,2})_with_fraction\.tif$", re.IGNORECASE)

def parse_year_month(path: Path):
    m = name_re.search(path.name)
    return (int(m.group(1)), int(m.group(2))) if m else None

def map_band_indices_by_name(ds: rio.DatasetReader):
    mapping = {}
    descs = ds.descriptions
    for i, d in enumerate(descs, start=1):
        if d is None: d = f"B{i}"
        mapping[_norm(d)] = i
    return mapping, descs

def compute_lonlat_grid(ds: rio.DatasetReader):
    h, w = ds.height, ds.width
    rows, cols = np.indices((h, w))
    xs, ys = rio.transform.xy(ds.transform, rows, cols, offset="center")
    x, y = np.asarray(xs, dtype=np.float64), np.asarray(ys, dtype=np.float64)
    if ds.crs is None: raise RuntimeError("Dataset has no CRS")
    if ds.crs.to_epsg() == 4326: return x.astype(np.float32), y.astype(np.float32)
    transformer = Transformer.from_crs(ds.crs, "EPSG:4326", always_xy=True)
    lon, lat = transformer.transform(x, y)
    return lon.astype(np.float32), lat.astype(np.float32)

def mode_ignore_nan(x: pd.Series):
    x = x.dropna()
    return x.value_counts().idxmax() if not x.empty else np.nan

def aggregate_to_coarse_grids_annual(
    year: int, ds: rio.DatasetReader, predictors_stack: np.ndarray,
    predictor_names: list, annual_frac: np.ndarray, lon: np.ndarray, lat: np.ndarray,
    grid_sizes_deg, burned_threshold, parquet_dir, coarse_tif_dir, coarse_shp_dir, base_name
):
    H, W = ds.height, ds.width
    N = H * W
    lon_flat, lat_flat, frac_flat = lon.ravel(), lat.ravel(), annual_frac.ravel()

    binary_4km_flat = np.zeros_like(frac_flat, dtype=np.uint8)
    valid_frac = ~np.isnan(frac_flat)
    binary_4km_flat[valid_frac & (frac_flat > 0)] = 1 

    pred_flat = {name: band.ravel() for name, band in zip(predictor_names, predictors_stack)}
    valid_idx = np.nonzero(valid_frac)[0]
    if valid_idx.size == 0: return

    frac_valid, bin_valid = frac_flat[valid_frac], binary_4km_flat[valid_frac]
    lon_valid, lat_valid = lon_flat[valid_frac], lat_flat[valid_frac]
    pred_valid = {name: arr[valid_frac] for name, arr in pred_flat.items()}

    for size_deg in grid_sizes_deg:
        # Check if parquet exists before aggregating
        parquet_path = parquet_dir / f"{base_name}_grid{size_deg}deg.parquet"
        if parquet_path.exists():
            print(f"[SKIP] Parquet exists: {parquet_path}")
            continue

        big_lon = size_deg * np.floor(lon_valid / size_deg)
        big_lat = size_deg * np.floor(lat_valid / size_deg)

        df_dict = {
            "big_lon": big_lon.astype(np.float32), "big_lat": big_lat.astype(np.float32),
            "burned_4km": bin_valid, "frac_4km": frac_valid, "flat_idx": valid_idx,
        }
        for name in predictor_names: df_dict[name] = pred_valid[name].astype(np.float32)

        df = pd.DataFrame(df_dict)
        agg_dict = {"burned_4km": "mean", "frac_4km": "mean"}
        for name in predictor_names:
            if name == "b1": agg_dict[name] = mode_ignore_nan
            elif name in ["relative_humidity", "total_precipitation_sum"]: agg_dict[name] = "min"
            elif name in ["temperature_2m", "temperature_2m_min", "temperature_2m_max", 
                        "build_up_index", "drought_code", "duff_moisture_code", 
                        "fine_fuel_moisture_code", "fire_weather_index", "initial_fire_spread_index"]:
                agg_dict[name] = "max"
            else: agg_dict[name] = "mean"

        grouped = df.groupby(["big_lon", "big_lat"], as_index=False).agg(agg_dict)
        grouped = grouped.rename(columns={"burned_4km": "burned_frac_4km"})
        grouped["burned_label"] = (grouped["burned_frac_4km"] >= burned_threshold).astype(np.uint8)
        grouped = grouped.sort_values(["big_lat", "big_lon"]).reset_index(drop=True)
        grouped["ID"] = np.arange(len(grouped), dtype=np.int64)
        grouped["year"], grouped["grid_deg"] = year, size_deg

        grouped.to_parquet(parquet_path, index=False)
        print(f"[PARQUET] Saved {parquet_path}")

        # --- GeoTIFF Output ---
        tif_path = coarse_tif_dir / f"{base_name}_grid{size_deg}deg_epsg4326_burned_unburned.tif"
        if not tif_path.exists():
            min_lon, max_lon = grouped["big_lon"].min(), grouped["big_lon"].max() + size_deg
            min_lat, max_lat = grouped["big_lat"].min(), grouped["big_lat"].max() + size_deg
            transform = from_origin(min_lon, max_lat, size_deg, size_deg)
            width, height = int(np.ceil((max_lon - min_lon) / size_deg)), int(np.ceil((max_lat - min_lat) / size_deg))

            shapes = [(Polygon([(l, t), (l+size_deg, t), (l+size_deg, t+size_deg), (l, t+size_deg)]), int(v)) 
                      for l, t, v in zip(grouped["big_lon"], grouped["big_lat"], grouped["burned_label"])]
            
            coarse_raster = rasterize(shapes=shapes, out_shape=(height, width), transform=transform, fill=255, dtype="uint8")
            profile = {"driver": "GTiff", "height": height, "width": width, "count": 1, "dtype": "uint8",
                       "crs": "EPSG:4326", "transform": transform, "nodata": 255, "compress": "LZW"}
            with rio.open(tif_path, "w", **profile) as dst: dst.write(coarse_raster, 1)
            print(f"[TIF] Saved {tif_path}")

        # --- Shapefile Output ---
        shp_path = coarse_shp_dir / f"{base_name}_grid{size_deg}deg_cells_epsg4326.shp"
        if not shp_path.exists():
            geoms = [Polygon([(l, t), (l+size_deg, t), (l+size_deg, t+size_deg), (l, t+size_deg)]) 
                     for l, t in zip(grouped["big_lon"], grouped["big_lat"])]
            shp_gdf = gpd.GeoDataFrame({"ID": grouped["ID"], "burned_label": grouped["burned_label"]}, 
                                        geometry=geoms, crs="EPSG:4326")
            shp_gdf.to_file(shp_path)
            print(f"[SHP] Saved {shp_path}")

# ----------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------
monthly_tifs = sorted(Path(OUT_DIR).glob("cems_e5l_firecci_*_with_fraction.tif"))
year_to_paths = defaultdict(list)
for p in monthly_tifs:
    ym = parse_year_month(p)
    if ym: year_to_paths[ym[0]].append((ym[1], p))

for year in sorted(year_to_paths.keys()):
    # SKIP logic: Check if the 1-deg Parquet already exists for this year
    expected_parquet = PARQUET_DIR / f"cems_e5l_firecci_{year}_annual_grid1deg.parquet"
    if expected_parquet.exists():
        print(f"[SKIP YEAR] All outputs for {year} appear to exist.")
        continue

    month_paths = sorted(year_to_paths[year], key=lambda x: x[0])
    print(f"\n[YEAR] {year} — Processing {len(month_paths)} files")

    with rio.open(month_paths[0][1]) as ds_template:
        H, W = ds_template.height, ds_template.width
        band_map, _ = map_band_indices_by_name(ds_template)
        
        predictor_indices, predictor_names = [], []
        for want_norm, want_orig in zip(WANTED_NORM, WANTED):
            if want_norm in band_map:
                predictor_indices.append(band_map[want_norm]); predictor_names.append(want_orig)
            else:
                for k_norm, idx in band_map.items():
                    if want_norm in k_norm or k_norm in want_norm:
                        predictor_indices.append(idx); predictor_names.append(want_orig); break
        
        frac_idx = band_map.get(FRACTION_NORM)
        if frac_idx is None: continue

        frac_months = []
        pred_months = {name: [] for name in predictor_names}

        for _, path in month_paths:
            with rio.open(path) as ds_m:
                for name, idx in zip(predictor_names, predictor_indices):
                    pred_months[name].append(ds_m.read(idx).astype(np.float32))
                frac_months.append(ds_m.read(frac_idx).astype(np.float32))

        annual_frac = np.nanmax(np.stack(frac_months), axis=0)
        predictor_arrays = [np.nanmean(np.stack(pred_months[n]), axis=0).astype(np.float32) for n in predictor_names]
        lon, lat = compute_lonlat_grid(ds_template)

        aggregate_to_coarse_grids_annual(
            year, ds_template, np.stack(predictor_arrays), predictor_names, annual_frac, lon, lat,
            GRID_SIZES_DEG, BURNED_THRESHOLD, PARQUET_DIR, COARSE_TIF_DIR, COARSE_SHP_DIR, f"cems_e5l_firecci_{year}_annual"
        )

print("\n[DONE]")

[SKIP YEAR] All outputs for 2001 appear to exist.
[SKIP YEAR] All outputs for 2002 appear to exist.
[SKIP YEAR] All outputs for 2003 appear to exist.
[SKIP YEAR] All outputs for 2004 appear to exist.
[SKIP YEAR] All outputs for 2005 appear to exist.
[SKIP YEAR] All outputs for 2006 appear to exist.
[SKIP YEAR] All outputs for 2007 appear to exist.
[SKIP YEAR] All outputs for 2008 appear to exist.
[SKIP YEAR] All outputs for 2009 appear to exist.
[SKIP YEAR] All outputs for 2010 appear to exist.
[SKIP YEAR] All outputs for 2011 appear to exist.
[SKIP YEAR] All outputs for 2012 appear to exist.
[SKIP YEAR] All outputs for 2013 appear to exist.
[SKIP YEAR] All outputs for 2014 appear to exist.
[SKIP YEAR] All outputs for 2015 appear to exist.
[SKIP YEAR] All outputs for 2016 appear to exist.
[SKIP YEAR] All outputs for 2017 appear to exist.
[SKIP YEAR] All outputs for 2018 appear to exist.
[SKIP YEAR] All outputs for 2019 appear to exist.
[SKIP YEAR] All outputs for 2020 appear to exist.


/explore/nobackup/people/spotter5/temp_dir/ipykernel_1458959/3126916826.py:215: RuntimeWarning: All-NaN slice encountered
  annual_frac = np.nanmax(np.stack(frac_months), axis=0)
/explore/nobackup/people/spotter5/temp_dir/ipykernel_1458959/3126916826.py:216: RuntimeWarning: Mean of empty slice
  predictor_arrays = [np.nanmean(np.stack(pred_months[n]), axis=0).astype(np.float32) for n in predictor_names]


[PARQUET] Saved /explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction/parquet_coarse_grids_annual_analytical/cems_e5l_firecci_2021_annual_grid1deg.parquet
[TIF] Saved /explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction/tifs_coarse_grids_annual_analytical/cems_e5l_firecci_2021_annual_grid1deg_epsg4326_burned_unburned.tif


/explore/nobackup/people/spotter5/temp_dir/ipykernel_1458959/3126916826.py:168: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  shp_gdf.to_file(shp_path)


[SHP] Saved /explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction/shp_coarse_grids_annual_analytical/cems_e5l_firecci_2021_annual_grid1deg_cells_epsg4326.shp

[YEAR] 2022 — Processing 12 files


/explore/nobackup/people/spotter5/temp_dir/ipykernel_1458959/3126916826.py:215: RuntimeWarning: All-NaN slice encountered
  annual_frac = np.nanmax(np.stack(frac_months), axis=0)
/explore/nobackup/people/spotter5/temp_dir/ipykernel_1458959/3126916826.py:216: RuntimeWarning: Mean of empty slice
  predictor_arrays = [np.nanmean(np.stack(pred_months[n]), axis=0).astype(np.float32) for n in predictor_names]


[PARQUET] Saved /explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction/parquet_coarse_grids_annual_analytical/cems_e5l_firecci_2022_annual_grid1deg.parquet
[TIF] Saved /explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction/tifs_coarse_grids_annual_analytical/cems_e5l_firecci_2022_annual_grid1deg_epsg4326_burned_unburned.tif
[SHP] Saved /explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction/shp_coarse_grids_annual_analytical/cems_e5l_firecci_2022_annual_grid1deg_cells_epsg4326.shp

[DONE]


/explore/nobackup/people/spotter5/temp_dir/ipykernel_1458959/3126916826.py:168: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  shp_gdf.to_file(shp_path)


In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Inspect all 1-degree coarse-grid parquet files across ALL years
and print global burned vs unburned statistics.
"""

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# PATH
# ------------------------------------------------------------------
# UPDATED: Pointing to the new 'analytical' directory containing the 0.05 threshold files
PARQUET_DIR = Path(
    "/explore/nobackup/people/spotter5/clelland_fire_ml/"
    "training_e5l_cems_firecci_with_fraction/parquet_coarse_grids_annual_analytical"
)

# ------------------------------------------------------------------
# FIND ALL 1-DEGREE FILES
# ------------------------------------------------------------------
if not PARQUET_DIR.exists():
    raise FileNotFoundError(f"Directory not found: {PARQUET_DIR}")

files_1deg = sorted(PARQUET_DIR.glob("*_grid1deg.parquet"))

print(f"\nLooking in: {PARQUET_DIR}")
print(f"Found {len(files_1deg)} 1-degree parquet files")

if not files_1deg:
    raise RuntimeError("No 1-degree parquet files found")

# ------------------------------------------------------------------
# READ + CONCAT
# ------------------------------------------------------------------
df_all = pd.concat(
    (pd.read_parquet(f) for f in files_1deg),
    ignore_index=True
).dropna()

# ------------------------------------------------------------------
# BASIC DATASET INFO
# ------------------------------------------------------------------
print("\n=== DATASET OVERVIEW (ALL YEARS, 1° GRID) ===")
print(f"Rows        : {len(df_all):,}")
print(f"Columns     : {df_all.shape[1]}")
print(f"Year range  : {int(df_all['year'].min())} → {int(df_all['year'].max())}")
print(f"Grid sizes  : {sorted(df_all['grid_deg'].unique().tolist())}")

# ------------------------------------------------------------------
# BURNED VS UNBURNED COUNTS
# ------------------------------------------------------------------
counts = df_all["burned_label"].value_counts().sort_index()

unburned = int(counts.get(0, 0))
burned   = int(counts.get(1, 0))
total    = unburned + burned

print("\n=== BURNED vs UNBURNED (ALL YEARS) ===")
print(f"Unburned (0): {unburned:,}")
print(f"Burned   (1): {burned:,}")
print(f"Total cells : {total:,}")

# ------------------------------------------------------------------
# RATIOS
# ------------------------------------------------------------------
print("\n=== RATIOS ===")
if burned > 0:
    print(f"Burned : Unburned   = 1 : {unburned / burned:.1f}")
    print(f"Unburned : Burned   = {unburned / burned:.1f} : 1")
else:
    print("No burned cells found.")

# ------------------------------------------------------------------
# PERCENTAGES
# ------------------------------------------------------------------
print("\n=== PERCENTAGES ===")
print(f"Burned   : {100 * burned / total:.3f}%")
print(f"Unburned : {100 * unburned / total:.3f}%")

# ------------------------------------------------------------------
# OPTIONAL DISTRIBUTION CHECKS
# ------------------------------------------------------------------
print("\n=== burned_frac_4km (ALL YEARS) ===")
print(df_all["burned_frac_4km"].describe())

if "frac_4km" in df_all.columns:
    print("\n=== frac_4km (mean annual fraction, ALL YEARS) ===")
    print(df_all["frac_4km"].describe())

# ------------------------------------------------------------------
# OPTIONAL: MISSINGNESS SNAPSHOT
# ------------------------------------------------------------------
print("\n=== TOP 15 MOST MISSING COLUMNS ===")
missing = df_all.isna().mean().sort_values(ascending=False)
print(missing.head(15))

print("\n[DONE] Global 1° coarse-grid statistics computed.")


Looking in: /explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction/parquet_coarse_grids_annual_analytical
Found 22 1-degree parquet files

=== DATASET OVERVIEW (ALL YEARS, 1° GRID) ===
Rows        : 111,999
Columns     : 23
Year range  : 2001 → 2022
Grid sizes  : [1]

=== BURNED vs UNBURNED (ALL YEARS) ===
Unburned (0): 102,907
Burned   (1): 9,092
Total cells : 111,999

=== RATIOS ===
Burned : Unburned   = 1 : 11.3
Unburned : Burned   = 11.3 : 1

=== PERCENTAGES ===
Burned   : 8.118%
Unburned : 91.882%

=== burned_frac_4km (ALL YEARS) ===
count    111999.000000
mean          0.031705
std           0.145727
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: burned_frac_4km, dtype: float64

=== frac_4km (mean annual fraction, ALL YEARS) ===
count    111999.000000
mean          0.021814
std           0.133574
min           0.000000
25%           0.000000
50%           0.000000
75%   

Stage 1 model

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Stage-1 LightGBM model (FASTER + FIXED for LightGBM 4.x):
Predict burned_label on 1-degree coarse-grid cells (EPSG:4326)

- Reads all *_grid1deg.parquet from 'analytical' folder across all years
- Uses selected predictor columns only
- Stratified K-Fold CV
- Randomized hyperparameter tuning (manual ParameterSampler)
- Optimizes for recall
- Uses early stopping via callbacks (LightGBM 4.x compatible)
- Finds optimal probability threshold (0.10–0.90) from OOF probabilities
- Saves model + metrics + plots to 'analytical' output folder
"""

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import joblib

import lightgbm as lgb
from lightgbm import LGBMClassifier

from sklearn.model_selection import StratifiedKFold, ParameterSampler
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix


# ---------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------
# UPDATED: Pointing to the new 'analytical' directory containing the 0.05 threshold files
PARQUET_DIR = Path(
    "/explore/nobackup/people/spotter5/clelland_fire_ml/"
    "training_e5l_cems_firecci_with_fraction/parquet_coarse_grids_annual_analytical"
)

# UPDATED: Output directory also has _analytical suffix
OUT_DIR = Path(
    "/explore/nobackup/people/spotter5/clelland_fire_ml/"
    "training_e5l_cems_firecci_with_fraction/stage_1_model_analytical"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
N_SPLITS = 5
RANDOM_STATE = 42

# How many random configs to try (reduce if needed)
N_ITER_SEARCH = 30

# IMPORTANT: avoid nested parallelism
# We'll run CV sequentially and let LightGBM use threads.
LGBM_THREADS = int(os.environ.get("SLURM_CPUS_PER_TASK", "0")) or os.cpu_count() or 8

# Early stopping rounds (LightGBM callbacks)
EARLY_STOPPING_ROUNDS = 200

THRESHOLDS = np.arange(0.10, 0.91, 0.10)

# Tuning subset control:
# - keep ALL positives
# - sample negatives up to this cap for tuning
NEG_CAP_FOR_TUNING = 300_000

FEATURES = [
    "DEM",
    "slope",
    "aspect",
    "b1",
    "relative_humidity",
    "total_precipitation_sum",
    "temperature_2m",
    "temperature_2m_min",
    "temperature_2m_max",
    "build_up_index",
    "drought_code",
    "duff_moisture_code",
    "fine_fuel_moisture_code",
    "fire_weather_index",
    "initial_fire_spread_index",
]
TARGET = "burned_label"


# ---------------------------------------------------------------------
# METRICS
# ---------------------------------------------------------------------
def iou_from_confusion(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    denom = tp + fp + fn
    return float(tp / denom) if denom > 0 else 0.0


def metrics_at_threshold(y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(np.uint8)
    return {
        "threshold": float(thr),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "iou": iou_from_confusion(y_true, y_pred),
    }


# ---------------------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------------------
def load_all_grid1deg(parquet_dir: Path) -> pd.DataFrame:
    files = sorted(parquet_dir.glob("*_grid1deg.parquet"))
    print(f"Looking in: {parquet_dir}")
    print(f"Found {len(files)} 1-degree parquet files")
    if not files:
        raise RuntimeError("No *_grid1deg.parquet files found")

    # Read only needed columns
    cols = FEATURES + [TARGET]
    dfs = [pd.read_parquet(f, columns=cols) for f in files]
    return pd.concat(dfs, ignore_index=True)


def prepare_xy(df: pd.DataFrame):
    df = df[FEATURES + [TARGET]].dropna(axis=0).copy()

    # categorical handling for b1
    df["b1"] = df["b1"].astype("Int64").astype("category")

    X = df[FEATURES]
    y = df[TARGET].astype(np.uint8).to_numpy()

    print("\nDataset size:", len(df))
    print("Class counts:", pd.Series(y).value_counts().to_dict())
    return X, y


def make_tuning_subset(X, y, neg_cap=NEG_CAP_FOR_TUNING, seed=RANDOM_STATE):
    """Keep all positives; cap negatives for tuning to speed up search."""
    rng = np.random.default_rng(seed)
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    if len(neg_idx) > neg_cap:
        neg_idx = rng.choice(neg_idx, size=neg_cap, replace=False)

    idx = np.concatenate([pos_idx, neg_idx])
    rng.shuffle(idx)

    Xs = X.iloc[idx].copy()
    ys = y[idx].copy()

    print(
        f"\n[TUNING SUBSET] positives={len(pos_idx):,}, "
        f"negatives_used={len(neg_idx):,}, total={len(idx):,}"
    )
    return Xs, ys


# ---------------------------------------------------------------------
# CV TRAIN/EVAL WITH EARLY STOPPING (LightGBM 4.x callbacks)
# ---------------------------------------------------------------------
def cv_oof_prob_with_params(X, y, params, cv, early_stopping_rounds=EARLY_STOPPING_ROUNDS):
    """
    Train one param set across folds with early stopping and return:
      - mean recall across folds at default 0.5 threshold (used for ranking)
      - OOF probabilities
    """
    oof_prob = np.zeros(len(y), dtype=np.float32)
    fold_recalls = []

    for fold, (tr, va) in enumerate(cv.split(X, y), start=1):
        X_tr, y_tr = X.iloc[tr], y[tr]
        X_va, y_va = X.iloc[va], y[va]

        model = LGBMClassifier(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="binary_logloss",
            categorical_feature=["b1"],
            callbacks=[
                lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=False)
            ],
        )

        prob = model.predict_proba(X_va)[:, 1].astype(np.float32)
        oof_prob[va] = prob

        pred_05 = (prob >= 0.5).astype(np.uint8)
        fold_recalls.append(recall_score(y_va, pred_05, zero_division=0))

        best_iter = getattr(model, "best_iteration_", None)
        print(f"  Fold {fold}: best_iter={best_iter} recall@0.5={fold_recalls[-1]:.4f}")

    return float(np.mean(fold_recalls)), oof_prob


# ---------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------
def main():
    df = load_all_grid1deg(PARQUET_DIR)
    X, y = prepare_xy(df)

    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    pos_weight = n_neg / max(n_pos, 1)

    print(f"Class imbalance neg/pos = {pos_weight:.1f}")
    print(f"Using LightGBM threads = {LGBM_THREADS}")
    print(f"LightGBM version = {lgb.__version__}")

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    # --- tune on subset for speed ---
    X_tune, y_tune = make_tuning_subset(X, y)

    # Base params (use many estimators + early stopping)
    base_params = dict(
        objective="binary",
        random_state=RANDOM_STATE,
        n_jobs=LGBM_THREADS,
        verbosity=-1,
        n_estimators=10_000,  # early stopping decides the true number of trees
    )

    # Smaller, effective search space
    param_dist = {
        "learning_rate": [0.01, 0.02, 0.03, 0.05],
        "num_leaves": [31, 63, 127, 255],
        "max_depth": [-1, 5, 7, 9],
        "min_child_samples": [10, 20, 40, 80],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "reg_lambda": [0.0, 0.1, 1.0, 5.0],
        "scale_pos_weight": [pos_weight * f for f in [0.5, 1, 2, 4]],
    }

    sampler = list(ParameterSampler(param_dist, n_iter=N_ITER_SEARCH, random_state=RANDOM_STATE))

    print("\n[TUNING] Starting manual random search with early stopping")
    best_score = -1.0
    best_params = None

    for i, p in enumerate(sampler, start=1):
        params = {**base_params, **p}
        print(f"\n  Config {i}/{N_ITER_SEARCH}: {p}")

        mean_recall, _ = cv_oof_prob_with_params(
            X_tune, y_tune,
            params=params,
            cv=cv,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        )

        print(f"  -> mean recall@0.5 (tuning subset): {mean_recall:.4f}")

        if mean_recall > best_score:
            best_score = mean_recall
            best_params = params

    if best_params is None:
        raise RuntimeError("Tuning failed to produce a best parameter set.")

    print("\n[BEST PARAMS]")
    print(json.dumps(best_params, indent=2))

    # --- Train final model (use 1 fold as early-stopping validation) ---
    tr, va = next(cv.split(X, y))
    X_tr, y_tr = X.iloc[tr], y[tr]
    X_va, y_va = X.iloc[va], y[va]

    final_model = LGBMClassifier(**best_params)
    final_model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="binary_logloss",
        categorical_feature=["b1"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False)
        ],
    )

    # Save model
    model_path = OUT_DIR / "lgbm_stage1_model.joblib"
    joblib.dump(final_model, model_path)

    # --- OOF probabilities on FULL data using best params ---
    print("\n[OOF] Computing out-of-fold probabilities on FULL dataset")
    _, oof_prob = cv_oof_prob_with_params(
        X, y,
        params=best_params,
        cv=cv,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )

    # Threshold sweep
    rows = [metrics_at_threshold(y, oof_prob, t) for t in THRESHOLDS]
    df_thr = pd.DataFrame(rows)

    best_row = (
        df_thr
        .sort_values(["recall", "precision", "f1"], ascending=False)
        .iloc[0]
    )

    # Save threshold metrics
    df_thr.to_csv(OUT_DIR / "threshold_metrics.csv", index=False)

    # Save metrics summary
    with open(OUT_DIR / "final_metrics.txt", "w") as f:
        f.write("Stage-1 LightGBM (1° grid)\n")
        f.write(json.dumps(best_row.to_dict(), indent=2))

    # Plot recall vs threshold
    plt.figure()
    plt.plot(df_thr["threshold"], df_thr["recall"], marker="o")
    plt.xlabel("Probability threshold")
    plt.ylabel("Recall")
    plt.title("Threshold vs Recall (OOF)")
    plt.grid(True)
    plt.savefig(OUT_DIR / "threshold_vs_recall.png", dpi=200, bbox_inches="tight")
    plt.close()

    print("\n=== BEST THRESHOLD ===")
    print(best_row)

    print(f"\nArtifacts saved to:\n{OUT_DIR}")
    print(f"Model saved to:\n{model_path}")


if __name__ == "__main__":
    main()

/home/spotter5/.conda/envs/xgboost_gpu/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Looking in: /explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction/parquet_coarse_grids_annual_analytical
Found 19 1-degree parquet files

Dataset size: 104291
Class counts: {0: 97740, 1: 6551}
Class imbalance neg/pos = 14.9
Using LightGBM threads = 10
LightGBM version = 4.5.0

[TUNING SUBSET] positives=6,551, negatives_used=97,740, total=104,291

[TUNING] Starting manual random search with early stopping

  Config 1/30: {'subsample': 0.6, 'scale_pos_weight': 14.919859563425431, 'reg_lambda': 0.0, 'num_leaves': 63, 'min_child_samples': 40, 'max_depth': -1, 'learning_rate': 0.02, 'colsample_bytree': 0.8}
  Fold 1: best_iter=12 recall@0.5=0.0000
  Fold 2: best_iter=12 recall@0.5=0.0000
  Fold 3: best_iter=12 recall@0.5=0.0000
  Fold 4: best_iter=12 recall@0.5=0.0000
  Fold 5: best_iter=12 recall@0.5=0.0000
  -> mean recall@0.5 (tuning subset): 0.0000

  Config 2/30: {'subsample': 1.0, 'scale_pos_weight': 29.839719126850863, 'reg_lambda': 5.0, 'num_leav

Now take that saved model and apply it to the parquet file and save a annual prediction of burned or unburned, and join it to the observed.  Make a new column which says the value is 2 if the model predicted it burned and it is observed burned, 1 if it the model predicted it is not burned but it was observed burned, 0 if they both say unburned and -1 if the model says it is burned but it is not burned.  Save as annual shapefiles. 

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Predict Stage-1 burned/unburned on annual 1° parquet (analytical), join to annual 1° shapefiles (analytical) by ID,
save annual shapefiles with TP/FN/TN/FP labels, AND build a per-year summary dataframe
with BOTH counts and percentages, plus a 4-panel percent plot (0–100% y-axis shared).

Percent is computed over VALID comparisons only (TP+FP+TN+FN), i.e. excludes "NA".
ALL OUTPUTS are suffixed with '_analytical' to prevent overwriting.
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import joblib

import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------
ROOT = Path(
    "/explore/nobackup/people/spotter5/clelland_fire_ml/"
    "training_e5l_cems_firecci_with_fraction"
)

# INPUTS (Analytical)
PARQUET_DIR = ROOT / "parquet_coarse_grids_annual_analytical"
OBS_SHP_DIR = ROOT / "shp_coarse_grids_annual_analytical"

# MODEL (Analytical)
MODEL_DIR   = ROOT / "stage_1_model_analytical"
MODEL_PATH  = MODEL_DIR / "lgbm_stage1_model.joblib"

THRESH_CSV  = MODEL_DIR / "threshold_metrics.csv"
THRESH_TXT  = MODEL_DIR / "final_metrics.txt"
DEFAULT_THRESHOLD = 0.5

# OUTPUTS (Analytical - suffixed to avoid overwrite)
OUT_SHP_DIR = MODEL_DIR / "pred_vs_obs_shapefiles_annual_analytical"
OUT_SHP_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = MODEL_DIR / "pred_obs_counts_and_pct_by_year_analytical.csv"
SUMMARY_PNG = MODEL_DIR / "pred_obs_percent_by_year_4panel_analytical.png"

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
FEATURES = [
    "DEM", "slope", "aspect", "b1", "relative_humidity",
    "total_precipitation_sum", "temperature_2m", "temperature_2m_min",
    "temperature_2m_max", "build_up_index", "drought_code",
    "duff_moisture_code", "fine_fuel_moisture_code", "fire_weather_index",
    "initial_fire_spread_index",
]

OBS_LABEL_CANDIDATES = [
    "burned_label", "burned_lab", "burn_label", "burned",
    "label", "obs_label", "class",
]

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
parq_re = re.compile(r"cems_e5l_firecci_(\d{4})_annual_grid1deg\.parquet$", re.IGNORECASE)
shp_re  = re.compile(r"cems_e5l_firecci_(\d{4})_annual_grid1deg_cells_epsg4326\.shp$", re.IGNORECASE)


def load_best_threshold() -> float:
    if THRESH_TXT.exists():
        try:
            txt = THRESH_TXT.read_text().splitlines()
            json_start = None
            for i, line in enumerate(txt):
                if line.strip().startswith("{"):
                    json_start = i
                    break
            if json_start is not None:
                import json
                d = json.loads("\n".join(txt[json_start:]))
                return float(d.get("threshold", DEFAULT_THRESHOLD))
        except Exception:
            pass

    if THRESH_CSV.exists():
        try:
            df = pd.read_csv(THRESH_CSV)
            df = df.sort_values(["recall", "precision", "f1"], ascending=False)
            return float(df.iloc[0]["threshold"])
        except Exception:
            pass

    return float(DEFAULT_THRESHOLD)


def ensure_b1_category(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    X["b1"] = X["b1"].astype("Int64").astype("category")
    return X


def find_year_parquets(parquet_dir: Path):
    out = {}
    for p in parquet_dir.glob("*_grid1deg.parquet"):
        m = parq_re.search(p.name)
        if m:
            out[int(m.group(1))] = p
    return dict(sorted(out.items()))


def find_year_shapefiles(shp_dir: Path):
    out = {}
    for p in shp_dir.glob("*.shp"):
        m = shp_re.search(p.name)
        if m:
            out[int(m.group(1))] = p
    return dict(sorted(out.items()))


def pick_obs_label_column(gdf: gpd.GeoDataFrame) -> str:
    cols = list(gdf.columns)
    cols_lower = {c.lower(): c for c in cols}

    for cand in OBS_LABEL_CANDIDATES:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]

    for c in cols:
        cl = c.lower()
        if ("burn" in cl and "lab" in cl) or cl in ("burn", "burned"):
            return c

    return ""


def label_tpfn_tnfp(obs: np.ndarray, pred: np.ndarray) -> np.ndarray:
    obs = obs.astype(np.uint8)
    pred = pred.astype(np.uint8)
    out = np.empty(obs.shape[0], dtype=object)
    out[(pred == 1) & (obs == 1)] = "TP"
    out[(pred == 0) & (obs == 1)] = "FN"
    out[(pred == 0) & (obs == 0)] = "TN"
    out[(pred == 1) & (obs == 0)] = "FP"
    return out


def plot_percent_4panel(df_counts: pd.DataFrame, out_png: Path):
    dfp = df_counts.sort_values("year").copy()
    years = dfp["year"].to_numpy()

    fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
    axes = axes.ravel()

    panels = [
        ("TP_pct", "TP (%)"),
        ("FN_pct", "FN (%)"),
        ("TN_pct", "TN (%)"),
        ("FP_pct", "FP (%)"),
    ]

    for ax, (col, title) in zip(axes, panels):
        ax.plot(years, dfp[col].to_numpy(), marker="o")
        ax.set_title(title)
        ax.set_xlabel("Year")
        ax.set_ylabel("Percent")
        ax.autoscale(enable=True, axis="y")
        ax.grid(True)

    plt.tight_layout()
    fig.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------
def main():
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

    model = joblib.load(MODEL_PATH)
    thr = load_best_threshold()

    print(f"[MODEL] {MODEL_PATH}")
    print(f"[THR]   {thr:.3f}")

    year_to_parq = find_year_parquets(PARQUET_DIR)
    year_to_shp  = find_year_shapefiles(OBS_SHP_DIR)

    years = sorted(set(year_to_parq) & set(year_to_shp))
    
    print(f"Looking for Parquets in: {PARQUET_DIR}")
    print(f"Looking for Shapefiles in: {OBS_SHP_DIR}")
    
    if not years:
        raise RuntimeError(
            "No overlapping years between parquet and shapefiles.\n"
            f"Parquet years found: {list(year_to_parq.keys())}\n"
            f"SHP years found: {list(year_to_shp.keys())}"
        )

    print(f"[YEARS] {years}")

    summary_rows = []

    for year in years:
        # Define output path first to check existence
        out_name = f"cems_e5l_firecci_{year}_annual_grid1deg_pred_vs_obs_analytical.shp"
        out_path = OUT_SHP_DIR / out_name

        print(f"\n=== {year} ===")
        
        # -----------------------------------------------------------------
        # CHECK: If output exists, load it to skip computation, but keep for summary
        # -----------------------------------------------------------------
        if out_path.exists():
            print(f"[RESUME] Found existing output: {out_path.name}")
            print(f"         Loading existing file to generate summary stats...")
            gdf = gpd.read_file(out_path)
            # Ensure proper types for summary logic
            if "year" not in gdf.columns:
                gdf["year"] = int(year)
        else:
            # -----------------------------------------------------------------
            # COMPUTE: Predict and Merge
            # -----------------------------------------------------------------
            parq_path = year_to_parq[year]
            shp_path  = year_to_shp[year]
            print(f"[PARQ] {parq_path.name}")
            print(f"[SHP ] {shp_path.name}")

            # --- predict on parquet ---
            dfp = pd.read_parquet(parq_path, columns=["ID"] + FEATURES).copy()
            dfp = dfp.dropna(subset=FEATURES).copy()

            X = ensure_b1_category(dfp[FEATURES])
            prob = model.predict_proba(X)[:, 1].astype(np.float32)
            pred = (prob >= thr).astype(np.uint8)

            pred_df = pd.DataFrame(
                {
                    "ID": dfp["ID"].astype(np.int64).to_numpy(),
                    "pred_prob": prob,
                    "pred_label": pred,
                }
            )

            # --- read observed shapefile ---
            gdf = gpd.read_file(shp_path)

            if "ID" not in gdf.columns:
                raise RuntimeError(f"[{year}] Shapefile missing 'ID' column: {shp_path}")

            obs_col = pick_obs_label_column(gdf)
            if not obs_col:
                raise RuntimeError(
                    f"[{year}] Could not find an observed label column in {shp_path}\n"
                    f"Available columns: {list(gdf.columns)}\n"
                    f"Tried candidates: {OBS_LABEL_CANDIDATES}"
                )
            print(f"[OBS] Using observed label column: '{obs_col}'")

            gdf["ID"] = gdf["ID"].astype(np.int64)

            obs_vals = pd.to_numeric(gdf[obs_col], errors="coerce")
            gdf["obs_label"] = obs_vals  # float with NaNs
            valid_obs = gdf["obs_label"].isin([0, 1])

            # --- join ---
            gdf = gdf.merge(pred_df, on="ID", how="left", validate="one_to_one")

            missing_pred = int(gdf["pred_label"].isna().sum())
            if missing_pred:
                print(f"[WARN] {missing_pred:,} polygons had no matching prediction by ID")

            # --- label TP/FN/TN/FP/NA ---
            gdf["pred_obs"] = "NA"
            valid = valid_obs & (~gdf["pred_label"].isna())

            if valid.any():
                gdf.loc[valid, "pred_label"] = gdf.loc[valid, "pred_label"].astype(np.uint8)
                gdf.loc[valid, "pred_obs"] = label_tpfn_tnfp(
                    gdf.loc[valid, "obs_label"].astype(np.uint8).to_numpy(),
                    gdf.loc[valid, "pred_label"].to_numpy(),
                )

            # ensure year column
            if "year" not in gdf.columns:
                gdf["year"] = int(year)

            # --- write shapefile (Analytical) ---
            gdf.to_file(out_path)
            print(f"[SAVE] {out_path}")

        # -----------------------------------------------------------------
        # COMMON: Calculate Stats (works for both Fresh and Loaded gdf)
        # -----------------------------------------------------------------
        vc = gdf["pred_obs"].value_counts().to_dict()
        tp = int(vc.get("TP", 0))
        fn = int(vc.get("FN", 0))
        tn = int(vc.get("TN", 0))
        fp = int(vc.get("FP", 0))
        na = int(vc.get("NA", 0))
        denom = tp + fn + tn + fp  # VALID comparisons only

        def pct(x):
            return float(100.0 * x / denom) if denom > 0 else 0.0

        row = {
            "year": int(year),
            "TP": tp,
            "FN": fn,
            "TN": tn,
            "FP": fp,
            "NA": na,
            "n_total": int(len(gdf)),
            "n_valid": int(denom),
            "TP_pct": pct(tp),
            "FN_pct": pct(fn),
            "TN_pct": pct(tn),
            "FP_pct": pct(fp),
        }
        summary_rows.append(row)

        print(f"[COUNTS] TP={tp:,} FN={fn:,} TN={tn:,} FP={fp:,} NA={na:,} (valid={denom:,})")
        print(f"[PCT]    TP={row['TP_pct']:.2f}% FN={row['FN_pct']:.2f}% TN={row['TN_pct']:.2f}% FP={row['FP_pct']:.2f}%")

    # -----------------------------------------------------------------
    # Save summary dataframe + plot percents (Analytical)
    # -----------------------------------------------------------------
    df_sum = pd.DataFrame(summary_rows).sort_values("year").reset_index(drop=True)
    df_sum.to_csv(SUMMARY_CSV, index=False)
    print(f"\n[SUMMARY] Saved CSV:  {SUMMARY_CSV}")

    plot_percent_4panel(df_sum[["year", "TP_pct", "FN_pct", "TN_pct", "FP_pct"]], SUMMARY_PNG)
    print(f"[PLOT]    Saved PNG:  {SUMMARY_PNG}")

    print(f"\n[DONE] Wrote {len(years)} annual shapefiles to {OUT_SHP_DIR}")


if __name__ == "__main__":
    main()

/home/spotter5/.conda/envs/xgboost_gpu/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[MODEL] /explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction/stage_1_model_analytical/lgbm_stage1_model.joblib
[THR]   0.100
Looking for Parquets in: /explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction/parquet_coarse_grids_annual_analytical
Looking for Shapefiles in: /explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction/shp_coarse_grids_annual_analytical
[YEARS] [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]

=== 2001 ===
[RESUME] Found existing output: cems_e5l_firecci_2001_annual_grid1deg_pred_vs_obs_analytical.shp
         Loading existing file to generate summary stats...
[COUNTS] TP=198 FN=8 TN=3,513 FP=1,770 NA=5,829 (valid=5,489)
[PCT]    TP=3.61% FN=0.15% TN=64.00% FP=32.25%

=== 2002 ===
[RESUME] Found existing output: cems_e5l_firecci_2002_annual_grid1deg_pred_vs_obs_analytical.s

Now take the cells we predicted as burnable and extract 4km predictor data per year and month and save to parquet file, and first print new ratio of burned to unburned.  Previously 1:4000, we want to see this imbalance drastically reduced. 

In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio as rio
from rasterio.features import rasterize
from rasterio.warp import transform as rio_transform
import geopandas as gpd
from tqdm import tqdm

import pyarrow as pa
import pyarrow.parquet as pq

# ================== CONFIG ==================
IN_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction")

# UPDATED: Point to the new analytical stage 1 model outputs (analytical shapefiles)
PRED_SHP_DIR = IN_DIR / "stage_1_model_analytical" / "pred_vs_obs_shapefiles_annual_analytical"

# UPDATED: Output directory for the new analytical dataset
OUT_DATASET_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/parquet_cems_with_fraction_dataset_burnedlab_mask_analytical")
OUT_DATASET_DIR.mkdir(parents=True, exist_ok=True)

REPROJECT_TO_EPSG4326 = True

# Years to process
YEAR_MIN = 2001
YEAR_MAX = 2022

# Mask shapefile criterion: keep only burned-lab cells
BURNED_LAB_VALUE = 1
BURNED_LAB_FIELD_OVERRIDE = None  # set if you know exact field name

# Fraction band description/name candidates (searched in ds.descriptions)
FRACTION_BAND_DESC_CANDIDATES = ["fraction", "frac", "burn_fraction"]

# Pixel label from fraction
PIXEL_BURN_THRESHOLD = 0.5  # burned if fraction > 0.5, unburned if fraction < 0.5

# ================== HELPERS ==================
def sanitize_names(names):
    """Make unique, safe column names (avoid duplicates)."""
    seen = {}
    out = []
    for n in names:
        if n is None or str(n).strip() == "":
            n = "band"
        n0 = re.sub(r"[^a-zA-Z0-9_]", "_", str(n).strip())
        n0 = re.sub(r"_+", "_", n0).strip("_")
        if n0 == "":
            n0 = "band"
        if n0 in seen:
            seen[n0] += 1
            n0 = f"{n0}_{seen[n0]}"
        else:
            seen[n0] = 1
        out.append(n0)
    return out

name_re = re.compile(r"cems_e5l_firecci_(\d{4})_(\d{1,2})_with_fraction\.tif$", re.IGNORECASE)
# Shapefile regex to match the new analytical naming convention
shp_re = re.compile(r"cems_e5l_firecci_(\d{4})_annual_grid1deg_pred_vs_obs_analytical\.shp$", re.IGNORECASE)

def parse_year_month(fname: str):
    m = name_re.search(fname)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))

def append_chunk_to_dataset(df: pd.DataFrame, root: Path):
    if not df.columns.is_unique:
        dups = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f"Duplicate column names found: {dups}")
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_to_dataset(
        table,
        root_path=str(root),
        partition_cols=["year", "month"],
        use_dictionary=False
    )

def find_fraction_band_index(ds: rio.DatasetReader) -> int:
    """
    Return 0-based band index for fraction band by inspecting ds.descriptions.
    """
    descs = list(ds.descriptions) if ds.descriptions else [None] * ds.count
    descs_safe = sanitize_names([d if d else f"B{i}" for i, d in enumerate(descs, start=1)])
    descs_safe_lower = [d.lower() for d in descs_safe]

    for cand in FRACTION_BAND_DESC_CANDIDATES:
        cand = cand.lower()
        for i, d in enumerate(descs_safe_lower):
            if cand == d or cand in d:
                return i

    raise RuntimeError(
        "Could not find fraction band by description. "
        f"Band descriptions (sanitized): {descs_safe}"
    )

def build_lonlat(ds: rio.DatasetReader, xs, ys):
    if (
        REPROJECT_TO_EPSG4326
        and ds.crs is not None
        and ds.crs.to_string().upper() not in ("EPSG:4326", "OGC:CRS84")
    ):
        lons, lats = rio_transform(ds.crs, "EPSG:4326", xs, ys)
        return np.asarray(lons, dtype=np.float64), np.asarray(lats, dtype=np.float64)
    return xs.astype(np.float64), ys.astype(np.float64)

def find_burned_lab_field(gdf: gpd.GeoDataFrame) -> str:
    """
    Find the 'burned_lab' field even if DBF truncates it.
    """
    if BURNED_LAB_FIELD_OVERRIDE:
        if BURNED_LAB_FIELD_OVERRIDE not in gdf.columns:
            raise RuntimeError(f"Override burned-lab field '{BURNED_LAB_FIELD_OVERRIDE}' not in: {list(gdf.columns)}")
        return BURNED_LAB_FIELD_OVERRIDE

    cols_lower = {c.lower(): c for c in gdf.columns}

    # common names
    candidates = ["burned_lab", "burned_label", "burnedlab", "burn_lab", "burnlab", "burned"]
    for c in candidates:
        if c in cols_lower:
            return cols_lower[c]

    # fuzzy fallback
    for c in gdf.columns:
        cl = c.lower()
        if "burn" in cl and ("lab" in cl or "label" in cl):
            return c

    raise RuntimeError(f"Could not find burned_lab field. Columns: {list(gdf.columns)}")

def raster_mask_from_burnedlab(ds: rio.DatasetReader, shp_path: Path) -> np.ndarray:
    """
    Rasterize polygons where burned_lab==1 onto ds grid -> boolean mask (H,W).
    """
    gdf = gpd.read_file(shp_path)
    lab_col = find_burned_lab_field(gdf)

    lab_vals = pd.to_numeric(gdf[lab_col], errors="coerce")
    gdf_keep = gdf.loc[lab_vals == BURNED_LAB_VALUE].copy()

    if gdf_keep.empty:
        return np.zeros((ds.height, ds.width), dtype=bool)

    if ds.crs is None:
        raise RuntimeError(f"Raster has no CRS; cannot rasterize: {shp_path}")
    if gdf_keep.crs is None:
        raise RuntimeError(f"Shapefile has no CRS; cannot rasterize: {shp_path}")

    if gdf_keep.crs != ds.crs:
        gdf_keep = gdf_keep.to_crs(ds.crs)

    shapes = [(geom, 1) for geom in gdf_keep.geometry if geom is not None and not geom.is_empty]
    if not shapes:
        return np.zeros((ds.height, ds.width), dtype=bool)

    mask_u8 = rasterize(
        shapes=shapes,
        out_shape=(ds.height, ds.width),
        transform=ds.transform,
        fill=0,
        dtype="uint8",
        all_touched=False,
    )
    return mask_u8.astype(bool)

# ================== MAIN ==================
def main():
    tifs = sorted(IN_DIR.glob("cems_e5l_firecci_*_with_fraction.tif"))
    if not tifs:
        raise FileNotFoundError(f"No monthly _with_fraction.tif found in {IN_DIR}")

    # Filter to years 2001-2019 only
    todo = []
    for tif in tifs:
        y, m = parse_year_month(tif.name)
        if y is None:
            continue
        if y < YEAR_MIN or y > YEAR_MAX:
            continue
        todo.append((y, m, tif))
    todo.sort()

    if not todo:
        raise RuntimeError(f"No TIFFs found in year range {YEAR_MIN}-{YEAR_MAX}")

    # Cache the rasterized burned-lab mask per year
    year_mask_cache = {}

    canonical_cols = None

    # Global ratio counters (only where burned_pixel is defined)
    burned_total = 0
    unburned_total = 0
    valid_lab_total = 0

    print(f"Scanning for analytical shapefiles in: {PRED_SHP_DIR}")

    for year, month, tif in tqdm(todo, desc="Building partitioned Parquet dataset (burned_lab mask)"):
        
        # ------------------------------------------------------------------
        # NEW: Check if output partition exists to avoid reprocessing
        # ------------------------------------------------------------------
        # PyArrow partitioning structure: root/year=YYYY/month=M
        partition_path = OUT_DATASET_DIR / f"year={year}" / f"month={month}"
        
        # Check directory existence AND ensure it contains at least one parquet file
        if partition_path.exists() and any(partition_path.glob("*.parquet")):
            # print(f"[SKIP] {year}-{month} exists.") # Uncomment for verbose skipping
            continue

        # ------------------------------------------------------------------
        # Prepare inputs
        # ------------------------------------------------------------------
        shp_name = f"cems_e5l_firecci_{year}_annual_grid1deg_pred_vs_obs_analytical.shp"
        shp_path = PRED_SHP_DIR / shp_name
        
        if not shp_path.exists():
            print(f"\n[SKIP] {tif.name} (missing annual analytical shapefile: {shp_path})")
            continue

        with rio.open(tif) as ds:
            # band names
            band_names = list(ds.descriptions) if ds.descriptions else []
            if not any(band_names):
                band_names = [f"B{i}" for i in range(1, ds.count + 1)]
            safe_names = sanitize_names(band_names)

            # fraction band index (0-based)
            frac_band0 = find_fraction_band_index(ds)
            frac_col_name = "fraction"

            # burned-lab mask per year (rasterized once)
            if year not in year_mask_cache:
                mask = raster_mask_from_burnedlab(ds, shp_path)
                year_mask_cache[year] = mask
                print(f"\n[YEAR {year}] burned_lab mask keeps {mask.sum():,} / {mask.size:,} pixels ({100*mask.mean():.2f}%)")
            else:
                mask = year_mask_cache[year]
                if mask.shape != (ds.height, ds.width):
                    raise RuntimeError(f"Mask shape mismatch for {year}: mask {mask.shape} vs raster {(ds.height, ds.width)}")

            if mask.sum() == 0:
                continue

            # Read raster (bands, H, W)
            data = ds.read().astype(np.float32)
            bands, h, w = data.shape

            # Flatten to (pixels, bands)
            arr2d = data.reshape(bands, -1).T

            # Keep only pixels with build_up_index not NaN (domain mask)
            build_col = None
            for s in safe_names:
                if "build" in s.lower() and "index" in s.lower():
                    build_col = s
                    break
            if build_col is None:
                raise ValueError(f"Could not find build_up_index band in: {tif.name}")

            build_idx = safe_names.index(build_col)
            build_vals = arr2d[:, build_idx]

            keep_mask = mask.reshape(-1) & (~np.isnan(build_vals))
            if not keep_mask.any():
                continue

            # Subset pixels
            arr_keep = arr2d[keep_mask, :]
            df = pd.DataFrame(arr_keep, columns=safe_names)

            # Ensure fraction column exists exactly once
            frac_vals_from_band = df.iloc[:, frac_band0].astype(np.float32).to_numpy()
            df[frac_col_name] = frac_vals_from_band  # overwrite if already present

            # burned_pixel binary from fraction
            frac_vals = df[frac_col_name].to_numpy(dtype=np.float32, copy=False)
            burned_pixel = np.full(frac_vals.shape, np.nan, dtype=np.float32)
            valid_frac = ~np.isnan(frac_vals)
            burned_pixel[valid_frac & (frac_vals > PIXEL_BURN_THRESHOLD)] = 1.0
            burned_pixel[valid_frac & (frac_vals < PIXEL_BURN_THRESHOLD)] = 0.0
            df["burned_pixel"] = burned_pixel

            # Update global counters
            valid_lab = ~np.isnan(burned_pixel)
            if valid_lab.any():
                burned_total += int(np.sum(burned_pixel[valid_lab] == 1.0))
                unburned_total += int(np.sum(burned_pixel[valid_lab] == 0.0))
                valid_lab_total += int(valid_lab.sum())

            # Coordinates for kept pixels
            rows = np.arange(h)
            cols = np.arange(w)
            rr, cc = np.meshgrid(rows, cols, indexing="ij")
            xs, ys = rio.transform.xy(ds.transform, rr, cc, offset="center")
            xs = np.asarray(xs, dtype=np.float64).reshape(-1)[keep_mask]
            ys = np.asarray(ys, dtype=np.float64).reshape(-1)[keep_mask]
            lons, lats = build_lonlat(ds, xs, ys)

            df["longitude"] = lons
            df["latitude"] = lats
            df["year"] = year
            df["month"] = month

            # Canonical schema
            if canonical_cols is None:
                canonical_cols = list(safe_names)
                if frac_col_name not in canonical_cols:
                    canonical_cols.append(frac_col_name)
                for extra in ["burned_pixel", "longitude", "latitude", "year", "month"]:
                    if extra not in canonical_cols:
                        canonical_cols.append(extra)
                if len(canonical_cols) != len(set(canonical_cols)):
                    raise RuntimeError(f"Canonical cols not unique: {canonical_cols}")

            for col in canonical_cols:
                if col not in df.columns:
                    df[col] = np.nan

            df = df[canonical_cols]
            append_chunk_to_dataset(df, OUT_DATASET_DIR)

    print(f"\n✅ Done. Parquet dataset at:\n{OUT_DATASET_DIR}\n(partitioned by year=/month=)")

    # Global ratios
    print("\n=== Burned/Unburned pixel counts (filtered to burned_lab==1 1° cells) ===")
    print(f"Valid labeled pixels (fraction != NaN and != {PIXEL_BURN_THRESHOLD}): {valid_lab_total:,}")
    print(f"Burned pixels    (fraction > {PIXEL_BURN_THRESHOLD}): {burned_total:,}")
    print(f"Unburned pixels (fraction < {PIXEL_BURN_THRESHOLD}): {unburned_total:,}")

    if burned_total > 0:
        ratio = unburned_total / burned_total
        print(f"Unburned:Burned ratio = {ratio:.3f} : 1")
    else:
        print("Unburned:Burned ratio = inf (no burned pixels found)")

if __name__ == "__main__":
    main()

Scanning for analytical shapefiles in: /explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction/stage_1_model_analytical/pred_vs_obs_shapefiles_annual_analytical


Building partitioned Parquet dataset (burned_lab mask):  81%|████████  | 213/264 [00:00<00:00, 343.63it/s]


[YEAR 2020] burned_lab mask keeps 357,310 / 4,273,642 pixels (8.36%)

[YEAR 2021] burned_lab mask keeps 461,220 / 4,273,642 pixels (10.79%)


Building partitioned Parquet dataset (burned_lab mask):  95%|█████████▌| 252/264 [00:42<00:06,  1.93it/s] 


[YEAR 2022] burned_lab mask keeps 414,397 / 4,273,642 pixels (9.70%)


Building partitioned Parquet dataset (burned_lab mask): 100%|██████████| 264/264 [01:05<00:00,  4.02it/s]


✅ Done. Parquet dataset at:
/explore/nobackup/people/spotter5/clelland_fire_ml/parquet_cems_with_fraction_dataset_burnedlab_mask_analytical
(partitioned by year=/month=)

=== Burned/Unburned pixel counts (filtered to burned_lab==1 1° cells) ===
Valid labeled pixels (fraction != NaN and != 0.5): 1,642,855
Burned pixels    (fraction > 0.5): 42,002
Unburned pixels (fraction < 0.5): 1,600,853
Unburned:Burned ratio = 38.114 : 1


In [ ]:
't'

Now lets train the stage two model but leave 2003 and 2004 out as the testing set.

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Train XGBoost - LEAVE-ONE-YEAR-OUT (LOYO)
-- Uses "Heavy Regularization" Parameters --
-- Iterates 2001-2022: Holds one year out as Test, trains on rest --
-- Uses base_score instead of scale_pos_weight for true probability outputs --
"""

import os
import sys
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
import pyarrow.dataset as ds
from sklearn.model_selection import train_test_split

# ============================================================
# CONFIG
# ============================================================

RANDOM_STATE = 42
DATASET_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/parquet_cems_with_fraction_dataset_burnedlab_mask_analytical")

# NEW OUTPUT DIRECTORY for LOYO results (Updated for base_score)
OUT_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_loyo_base_score")

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "models").mkdir(exist_ok=True)

LOG_FILE = OUT_DIR / "training_log_loyo.txt"

# FEATURES
FEATURES = [
    "DEM", "slope", "aspect", "b1", "relative_humidity",
    "total_precipitation_sum", "temperature_2m", "temperature_2m_min",
    "temperature_2m_max", "build_up_index", "drought_code",
    "duff_moisture_code", "fine_fuel_moisture_code",
    "fire_weather_index", "initial_fire_spread_index",
]

FRACTION_COL = "fraction"
LABEL_COL = "burned"
VAL_SIZE_OF_REMAINING = 0.20 

# TRAINING CONFIG
FINAL_ROUNDS  = 3000            
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", "0")) or os.cpu_count() or 8
USE_GPU = bool(os.environ.get("CUDA_VISIBLE_DEVICES", "").strip())
TREE_METHOD = "gpu_hist" if USE_GPU else "hist"

# ============================================================
# HELPERS
# ============================================================

class Logger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w", encoding='utf-8')
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()
    def flush(self):
        self.terminal.flush()
        self.log.flush()

def find_area_match_threshold(y_true, y_probs):
    n_burned = np.sum(y_true)
    n_total = len(y_true)
    if n_burned == 0:
        return 0.99 # Fallback if no burning
    target_percentile = 100.0 * (1.0 - (n_burned / n_total))
    return float(np.percentile(y_probs, target_percentile))

def prepare_df_cleaned(df: pd.DataFrame):
    df = df.copy()
    
    # 1. Types & Fraction
    df[FRACTION_COL] = pd.to_numeric(df[FRACTION_COL], errors="coerce").astype("float32")
    df = df[df[FRACTION_COL].notna() & (df[FRACTION_COL] != 0.5)].copy()
    df[LABEL_COL] = (df[FRACTION_COL] > 0.5).astype("uint8")
    
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["b1"] = pd.to_numeric(df["b1"], errors="coerce").round().astype("Int64")

    # 2. Filter Negative FWI
    fwi_cols = ["duff_moisture_code", "drought_code", "fine_fuel_moisture_code", "build_up_index"]
    for c in fwi_cols:
        if c in df.columns:
            df = df[df[c] >= 0]

    # 3. NaNs
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=FEATURES + [LABEL_COL, "year"])
    
    for c in FEATURES:
        if c == "b1": continue
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")
    df["b1"] = df["b1"].astype("int32")
        
    return df.copy()

def load_data():
    print(f"Loading Dataset from: {DATASET_DIR}")
    dset = ds.dataset(str(DATASET_DIR), format="parquet", partitioning="hive")
    cols = FEATURES + [FRACTION_COL, "year"]
    available_cols = dset.schema.names
    cols_to_load = [c for c in cols if c in available_cols]
    if "year" not in cols_to_load: cols_to_load.append("year")
    
    table = dset.to_table(columns=cols_to_load)
    df = table.to_pandas()
    return prepare_df_cleaned(df)

# ============================================================
# MAIN LOOP
# ============================================================

def main():
    # 1. Setup Logging
    sys.stdout = Logger(str(LOG_FILE))
    print(f"Logging initialized. Writing to: {LOG_FILE}")
    print("-" * 50)
    print(f"Starting Leave-One-Year-Out Training (GPU={USE_GPU})")
    
    # 2. Load Data Once
    df_all = load_data()
    all_years = sorted(df_all["year"].unique())
    print(f"Total Rows: {len(df_all):,}")
    print(f"Years found: {all_years}")
    
    results = []

    # 3. Iterate through each year
    for test_year in all_years:
        print("\n" + "#" * 60)
        print(f"PROCESSING YEAR: {test_year} (Test Set)")
        print("#" * 60)

        # --- A. Split Data ---
        mask_test = df_all["year"] == test_year
        df_test = df_all[mask_test]
        df_tv = df_all[~mask_test] # Train + Val

        if len(df_test) == 0:
            print(f"Skipping {test_year} (No data found).")
            continue

        X_tv = df_tv[FEATURES]
        y_tv = df_tv[LABEL_COL].values
        
        # Split remaining years into Train / Validation
        X_train, X_val, y_train, y_val = train_test_split(
            X_tv, y_tv, test_size=VAL_SIZE_OF_REMAINING, 
            random_state=RANDOM_STATE, stratify=y_tv
        )

        print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test ({test_year}): {len(df_test):,}")

        # --- B. Calculate Base Score ---
        # Calculate the global mean of the training labels for initialization
        base_rate = float(y_train.sum() / len(y_train))
        print(f"Base Score (Prior Probability of Burn): {base_rate:.6f}")

        # --- C. Train Model (Heavy Regularization) ---
        dtrain = xgb.DMatrix(X_train, label=y_train, nthread=N_JOBS)
        dval   = xgb.DMatrix(X_val,   label=y_val,   nthread=N_JOBS)

        final_params = {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "tree_method": TREE_METHOD,
            "max_bin": 256,
            "seed": RANDOM_STATE,
            "learning_rate": 0.05,
            "base_score": base_rate,  # Natively handles the class imbalance
            
            # --- OPTIMIZED REGULARIZATION PARAMS ---
            "max_depth": 4,           
            "min_child_weight": 100,  
            "gamma": 5.0,             
            "subsample": 0.5,         
            "colsample_bytree": 0.5,  
            "reg_lambda": 10.0,       
            "reg_alpha": 1.0,         
        }

        booster = xgb.train(
            params=final_params,
            dtrain=dtrain,
            num_boost_round=int(FINAL_ROUNDS),
            evals=[(dval, "val")],
            early_stopping_rounds=200,
            verbose_eval=500 
        )

        # --- D. Save Model ---
        model_name = f"xgb_loyo_{test_year}.json"
        save_path = OUT_DIR / "models" / model_name
        booster.save_model(str(save_path))
        print(f"Model saved: {save_path}")

        # --- E. Evaluate ---
        # 1. Validation Threshold (from other years)
        val_probs = booster.predict(dval)
        val_thr = find_area_match_threshold(y_val, val_probs)
        
        # 2. Test Predictions (current year)
        dtest = xgb.DMatrix(df_test[FEATURES], label=df_test[LABEL_COL].values, nthread=N_JOBS)
        test_probs = booster.predict(dtest)
        y_test = df_test[LABEL_COL].values
        
        # 3. Metrics
        obs_count = np.sum(y_test)
        pred_bin_count = np.sum((test_probs >= val_thr).astype(np.uint8))
        pred_sum_prob = np.sum(test_probs)
        
        diff_bin = pred_bin_count - obs_count
        oracle_thr = find_area_match_threshold(y_test, test_probs)

        row = {
            "test_year": int(test_year),
            "obs_pixels": int(obs_count),
            "pred_pixels_bin": int(pred_bin_count),
            "pred_pixels_sum": float(pred_sum_prob),
            "diff_bin": int(diff_bin),
            "error_pct": float(diff_bin / obs_count) if obs_count > 0 else 0.0,
            "val_threshold": float(val_thr),
            "oracle_threshold": float(oracle_thr),
            "model_path": str(model_name)
        }
        results.append(row)

        print(f"Results {test_year}: Obs={obs_count:,}, Pred={pred_bin_count:,}, Thr={val_thr:.4f}")
        
        # Cleanup to save memory
        del dtrain, dval, dtest, booster
        gc.collect()

    # 4. Save Summary
    print("\n" + "="*50)
    print("LOYO SUMMARY")
    print("="*50)
    
    df_results = pd.DataFrame(results)
    csv_path = OUT_DIR / "loyo_metrics_summary.csv"
    df_results.to_csv(csv_path, index=False)
    
    print(df_results[["test_year", "obs_pixels", "pred_pixels_bin", "error_pct", "val_threshold"]])
    print(f"\nAll Done. Summary saved to {csv_path}")

if __name__ == "__main__":
    main()

Logging initialized. Writing to: /explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_loyo_base_score/training_log_loyo.txt
--------------------------------------------------
Starting Leave-One-Year-Out Training (GPU=True)
Loading Dataset from: /explore/nobackup/people/spotter5/clelland_fire_ml/parquet_cems_with_fraction_dataset_burnedlab_mask_analytical
Total Rows: 75,674,335
Years found: [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]

############################################################
PROCESSING YEAR: 2001 (Test Set)
############################################################
Train: 58,968,226 | Val: 14,742,057 | Test (2001): 1,964,052
Base Score (Prior Probability of Burn): 0.003108


/home/spotter5/.conda/envs/xgboost_gpu/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [12:25:13] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1724807734561/work/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


[0]	val-logloss:0.02045
[500]	val-logloss:0.01520
[1000]	val-logloss:0.01469
[1500]	val-logloss:0.01440
[2000]	val-logloss:0.01418
[2500]	val-logloss:0.01403
[2999]	val-logloss:0.01391
Model saved: /explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_loyo_base_score/models/xgb_loyo_2001.json


/home/spotter5/.conda/envs/xgboost_gpu/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [12:28:31] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1724807734561/work/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/explore/nobackup/people/spotter5/temp_dir/ipykernel_1280581/782887889.py:224: RuntimeWarning: overflow encountered in scalar subtract
  diff_bin = pred_bin_count - obs_count


Results 2001: Obs=5,361, Pred=2,484, Thr=0.0852

############################################################
PROCESSING YEAR: 2002 (Test Set)
############################################################
Train: 57,259,839 | Val: 14,314,960 | Test (2002): 4,099,536
Base Score (Prior Probability of Burn): 0.003105


...

Results 2021: Obs=22,299, Pred=1,046, Thr=0.0822

############################################################
PROCESSING YEAR: 2022 (Test Set)
############################################################
Train: 60,394,196 | Val: 15,098,550 | Test (2022): 181,589
Base Score (Prior Probability of Burn): 0.002906


/home/spotter5/.conda/envs/xgboost_gpu/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [13:53:57] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1724807734561/work/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


[0]	val-logloss:0.01929
[500]	val-logloss:0.01426
[1000]	val-logloss:0.01378
[1500]	val-logloss:0.01347
[2000]	val-logloss:0.01327
[2500]	val-logloss:0.01312
[2999]	val-logloss:0.01299
Model saved: /explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_loyo_base_score/models/xgb_loyo_2022.json


/home/spotter5/.conda/envs/xgboost_gpu/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [13:57:18] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1724807734561/work/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/explore/nobackup/people/spotter5/temp_dir/ipykernel_1280581/782887889.py:224: RuntimeWarning: overflow encountered in scalar subtract
  diff_bin = pred_bin_count - obs_count


Results 2022: Obs=15,056, Pred=122, Thr=0.0879

LOYO SUMMARY
    test_year  obs_pixels  pred_pixels_bin     error_pct  val_threshold
0        2001        5361             2484  3.440915e+15       0.085165
1        2002       12198             6867  1.512276e+15       0.086785
2        2003       28254             8730  6.528896e+14       0.082154
3        2004        6735             4431  2.738938e+15       0.086957
4        2005        6621             4293  2.786096e+15       0.085760
5        2006        5688             1506  3.243098e+15       0.086109
6        2007        4887             2973  3.774656e+15       0.085623
7        2008       12207             4980  1.511161e+15       0.083399
8        2009        5421             5274  3.402830e+15       0.085829
9        2010        7053             3045  2.615446e+15       0.086235
10       2011        7779            16230  1.086386e+00       0.087787
11       2012       15084             9543  1.222935e+15       0.086560
12 

In [5]:
't'

't'

Now lets take that model, and get pixels which were flagged as TP and FP and predict on only those pixels. 

In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Step 1: Predict True Probabilities (LOYO Strategy)
-- Loads specific model for each year trained with base_score --
-- Predicts on that specific year (the held-out test set) --
-- Outputs natively calibrated true probabilities to TIFF --
"""

import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from rasterio import features
import xgboost as xgb

# ============================================================
# CONFIG
# ============================================================

# INPUTS
SHP_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction/stage_1_model_analytical/pred_vs_obs_shapefiles_annual_analytical")
IN_TIF_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction")

# DIR CONTAINING NEW LOYO MODELS (Updated to base_score)
MODELS_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_loyo_base_score/models")

# NEW OUTPUT DIRECTORY
OUT_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/predictions_loyo_base_score")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "DEM", "slope", "aspect", "b1", "relative_humidity", 
    "total_precipitation_sum", "temperature_2m", "temperature_2m_min", 
    "temperature_2m_max", "build_up_index", "drought_code", 
    "duff_moisture_code", "fine_fuel_moisture_code", 
    "fire_weather_index", "initial_fire_spread_index"
]

# ============================================================
# MAIN
# ============================================================

def main():
    # 1. Identify Years to Process based on Shapefiles
    shp_files = list(SHP_DIR.glob("*_analytical.shp"))
    years = sorted([int(re.findall(r'firecci_(\d{4})_', f.name)[0]) for f in shp_files if re.search(r'firecci_(\d{4})_', f.name)])
    
    if not years:
        print(f"No shapefiles found in {SHP_DIR}")
        return

    print(f"Found {len(years)} years to process: {years}")
    
    # 2. Iterate Per Year
    for year in years:
        print(f"\n" + "="*50)
        print(f"STARTING YEAR: {year}")
        print("="*50)

        # --- A. Load the Specific LOYO Model ---
        model_path = MODELS_DIR / f"xgb_loyo_{year}.json"
        
        if not model_path.exists():
            print(f"⚠️  Model not found for {year}: {model_path}")
            print("Skipping this year...")
            continue
            
        print(f"Loading Model: {model_path.name}")
        booster = xgb.Booster()
        booster.load_model(str(model_path))

        # --- B. Load Shapefile Mask ---
        shp_path = SHP_DIR / f"cems_e5l_firecci_{year}_annual_grid1deg_pred_vs_obs_analytical.shp"
        if not shp_path.exists(): 
            print("Shapefile missing.")
            continue
        
        gdf = gpd.read_file(shp_path)
        
        if 'pred_obs' not in gdf.columns: 
            print("Column 'pred_obs' missing.")
            continue
            
        # Filter for TP and FP zones (Areas where we want predictions)
        # Note: If you want to predict EVERYWHERE, remove this filter. 
        # But usually we only want the 'coarse' grid cells identified as potential fire.
        mask_gdf = gdf[gdf['pred_obs'].str.upper().isin(['TP', 'FP'])].copy()
        
        if mask_gdf.empty:
            print(f"No TP/FP regions found for {year}. Skipping.")
            continue

        print(f"Processing {len(mask_gdf)} TP/FP coarse cells...")

        # --- C. Iterate Months ---
        for month in range(1, 13):
            # Input Tif
            tif_path = IN_TIF_DIR / f"cems_e5l_firecci_{year}_{month}_with_fraction.tif"
            if not tif_path.exists(): continue
            
            # Output Tif
            out_name = OUT_DIR / f"pred_loyo_{year}_{month:02d}.tif"
            
            if out_name.exists():
                print(f"  Month {month:02d} already exists. Skipping.")
                continue

            try:
                with rio.open(tif_path) as src:
                    # 1. Align CRS
                    if mask_gdf.crs != src.crs: 
                        mask_gdf_proj = mask_gdf.to_crs(src.crs)
                    else:
                        mask_gdf_proj = mask_gdf

                    # 2. Rasterize Mask
                    # Burn 1s where TP/FP polygons are, 0 elsewhere
                    mask = features.rasterize(
                        [(geom, 1) for geom in mask_gdf_proj.geometry],
                        out_shape=src.shape, 
                        transform=src.transform, 
                        fill=0, 
                        dtype='uint8'
                    )
                    
                    # Optimization: If the mask is empty, skip reading the big raster
                    if not np.any(mask == 1): 
                        continue

                    # 3. Read Data
                    # Read all bands (assuming band order matches FEATURES list index logic if strictly positional, 
                    # but usually we extract by index logic below)
                    img_data = src.read()
                    nodata_val = src.nodata

                    # Get indices of pixels inside the mask
                    idx_y, idx_x = np.where(mask == 1)
                    
                    # Extract pixels: shape (n_features, n_pixels) -> transpose to (n_pixels, n_features)
                    pixels = img_data[:, idx_y, idx_x].T
                    
                    # Ensure we have enough features (15 based on list)
                    if pixels.shape[1] < len(FEATURES): 
                        print(f"  Month {month}: Band count mismatch. Expected >={len(FEATURES)}, got {pixels.shape[1]}")
                        continue
                        
                    # Slice exactly the features we need
                    pixels = pixels[:, :len(FEATURES)].astype('float32')
                    
                    # 4. Handle Missing Values / Nodata
                    if nodata_val is not None: 
                        pixels[pixels == nodata_val] = np.nan
                    
                    # Replace large negatives often found in met data with nan
                    pixels[pixels < -9000] = np.nan 

                    # 5. Predict using LOYO Model
                    # XGBoost handles NaNs natively
                    dmat = xgb.DMatrix(pixels, feature_names=FEATURES)
                    preds = booster.predict(dmat)
                    
                    # 6. Write Result
                    out_proba = np.zeros((src.height, src.width), dtype='float32')
                    out_proba[idx_y, idx_x] = preds
                    
                    out_meta = src.meta.copy()
                    out_meta.update(
                        dtype='float32', 
                        count=1, 
                        nodata=0, 
                        compress='deflate'
                    )
                    
                    with rio.open(out_name, 'w', **out_meta) as dst:
                        dst.write(out_proba, 1)
                    
                    print(f"  Saved: {out_name.name} (Predicted {len(preds):,} pixels)")

            except Exception as e:
                print(f"  Failed to process {year}-{month:02d}: {e}")

    print("\n✅ All available years processed with LOYO base_score models.")

if __name__ == "__main__":
    main()

In [8]:
't'

't'

Save annual tifs now

In [10]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Step 4: Create Annual Binary Burn Maps (LOYO Base Score - Fixed Threshold)
-- Reads Monthly True Probabilities (LOYO Predictions) --
-- Applies a FIXED threshold of 0.085 across all years --
-- Aggregates to Annual (Logical OR) --
-- Saves Annual Binary TIFFs --
"""

import os
import numpy as np
import rasterio as rio
from pathlib import Path
from tqdm import tqdm

# ============================
# CONFIG
# ============================

# Input Directory (LOYO Predictions from previous step)
IN_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/predictions_loyo_base_score")

# Output Directory (For Annual Binary Tifs)
OUT_DIR = IN_DIR / "annual_binary_masks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# FIXED THRESHOLD
PROB_THRESHOLD = 0.085

# Years to process
YEARS = list(range(2001, 2024))

# ============================
# MAIN
# ============================

def get_annual_mask(year, pred_dir, threshold):
    """
    Aggregates monthly probability TIFFs into a single annual binary mask.
    Returns: mask (numpy), profile (dict)
    """
    annual_mask = None
    profile = None
    found_files = False

    # Iterate months 1-12
    for month in range(1, 13):
        # Pattern matches the LOYO output (pred_loyo_YYYY_MM.tif)
        tif_path = pred_dir / f"pred_loyo_{year}_{month:02d}.tif"
        
        if not tif_path.exists():
            continue
        
        found_files = True
        
        with rio.open(tif_path) as src:
            # On first found file, grab profile/metadata
            if profile is None:
                profile = src.profile.copy()
                # Update profile for binary output (0/1)
                profile.update(
                    dtype='uint8',
                    count=1,
                    nodata=0,
                    compress='deflate'
                )
                # Initialize accumulator
                annual_mask = np.zeros((src.height, src.width), dtype=bool)

            # Read Probability
            prob_data = src.read(1)
            
            # Apply FIXED Threshold
            monthly_burn = (prob_data >= threshold)
            
            # Logical OR: Aggregate burns across months
            annual_mask = annual_mask | monthly_burn

    if not found_files:
        return None, None
        
    return annual_mask, profile

def main():
    if not IN_DIR.exists():
        raise FileNotFoundError(f"Input directory not found: {IN_DIR}")

    print(f"Input Directory: {IN_DIR}")
    print(f"Output Directory: {OUT_DIR}")
    print(f"Applying FIXED Threshold: {PROB_THRESHOLD} to all years.")

    for year in tqdm(YEARS, desc="Generating Annual Maps"):
        
        # Generate the aggregated mask using the fixed threshold
        mask, profile = get_annual_mask(year, IN_DIR, PROB_THRESHOLD)
        
        if mask is None:
            # print(f"Skipping {year} (No monthly files found)") 
            continue
            
        # Define output filename
        out_name = OUT_DIR / f"loyo_base_score_annual_binary_{year}.tif"
        
        # Save to disk
        # Convert boolean mask to uint8 (0 and 1)
        out_data = mask.astype('uint8')
        
        with rio.open(out_name, 'w', **profile) as dst:
            dst.write(out_data, 1)
            
        tqdm.write(f"Saved: {out_name.name} | Threshold: {PROB_THRESHOLD:.3f} | Burned Pixels: {np.sum(out_data):,}")

    print("\n✅ Annual processing complete.")

if __name__ == "__main__":
    main()

Generating Annual Maps: 100%|██████████| 23/23 [00:09<00:00,  2.45it/s]


Make maps of comparison for each year using epsg 3413 crs.  make sure to clip to study domain

In [11]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Generate 1x3 Annual Burned Area Maps (Hectares per Grid Cell).
-- Summarizes pixel-level burns into polygon-level area (Ha) --
-- Uses 'pred_grid_clip_small.shp' as the aggregation grid --
-- Scales all 3 plots (Pred, FireCCI, MCD) to the same color range --
"""

import os
import numpy as np
import rasterio as rio
from rasterio.features import rasterize
from rasterio.warp import reproject, Resampling
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import make_axes_locatable
from pathlib import Path
from tqdm import tqdm
import scipy.ndimage

# Set non-interactive backend for server environments
plt.switch_backend('Agg')

# ============================
# CONFIG
# ============================

# INPUT DIRECTORIES (Updated to LOYO Base Score)
PRED_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/predictions_loyo_base_score/annual_binary_masks")

FIRECCI_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction")
MCD_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_mcd_with_fraction")

# GRID SHAPEFILE (Aggregation Unit)
GRID_SHP_PATH = Path("/explore/nobackup/people/spotter5/cnn_mapping/raw_files/pred_grid_clip_small.shp")
GRID_ID_COL = "Id"  # Unique identifier in the shapefile

# OUTPUT (New directory for base score plots)
OUT_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/comparison_plots_grid_hectares_loyo_base_score")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# THRESHOLD (For Reference Data Fraction Band)
FRACTION_THRESHOLD = 0.5

# ============================
# HELPER FUNCTIONS
# ============================

def get_pixel_area_hectares(shape, transform, crs):
    """
    Creates a grid where every pixel value is its area in Hectares.
    """
    height, width = shape
    res_x = abs(transform.a)
    res_y = abs(transform.e)

    if crs.is_geographic:
        # Latitude dependent area calculation
        # 1 deg lat ~ 111,320 m
        rows = np.arange(height) + 0.5
        _, lats = rio.transform.xy(transform, rows, np.zeros_like(rows), offset='center')
        lats = np.array(lats)
        
        # Calculate width of 1 deg longitude at this latitude
        lat_rads = np.radians(lats)
        pixel_width_m = res_x * 111320 * np.cos(lat_rads)
        pixel_height_m = res_y * 111320
        
        # Area in m2
        row_areas_m2 = pixel_width_m * pixel_height_m
        
        # Broadcast to full grid
        area_grid_m2 = row_areas_m2[:, np.newaxis] * np.ones((1, width))
    else:
        # Projected CRS (meters)
        pixel_area_m2 = res_x * res_y
        area_grid_m2 = np.full(shape, pixel_area_m2)

    # Convert m2 to Hectares (1 Ha = 10,000 m2)
    return (area_grid_m2 / 10000.0).astype('float32')

def get_annual_reference_mask(year, source_dir, file_pattern_func, target_profile):
    """
    Aggregates monthly reference files into annual binary mask matched to target grid.
    Reads the LAST BAND (assumed to be 'fraction').
    """
    annual_mask = None
    height, width = target_profile['height'], target_profile['width']
    
    found_any = False
    for month in range(1, 13):
        tif_path = source_dir / file_pattern_func(year, month)
        if not tif_path.exists(): continue
        found_any = True
        
        with rio.open(tif_path) as src:
            # "fraction" is the last band. Rasterio is 1-indexed.
            fraction_band_idx = src.count 
            
            # Read the last band, reprojecting on the fly
            dest = np.zeros((height, width), dtype='float32')
            reproject(
                source=rio.band(src, fraction_band_idx),
                destination=dest,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=target_profile['transform'],
                dst_crs=target_profile['crs'],
                resampling=Resampling.nearest
            )
            # Threshold > 0.5
            mask = (dest > FRACTION_THRESHOLD)
            
            if annual_mask is None:
                annual_mask = mask
            else:
                annual_mask = annual_mask | mask # Logical OR

    if not found_any or annual_mask is None:
        return np.zeros((height, width), dtype=bool) 
        
    return annual_mask

def summarize_burn_per_grid(binary_mask, area_grid, grid_id_raster, unique_ids):
    """
    Sums the area of burned pixels within each grid ID.
    Returns: Dictionary {Grid_ID: Burned_Ha}
    """
    # Calculate burned area for every pixel (0 if not burned, area_ha if burned)
    burned_area_pixel = np.where(binary_mask, area_grid, 0)
    
    # Sum area by Grid ID using fast scipy routine
    # index=unique_ids ensures the output array matches the order of unique_ids
    sums = scipy.ndimage.sum(burned_area_pixel, labels=grid_id_raster, index=unique_ids)
    
    return dict(zip(unique_ids, sums))

# ============================
# MAIN
# ============================

def main():
    # 1. Load Grid Shapefile
    print(f"Loading Grid: {GRID_SHP_PATH}")
    gdf_grid = gpd.read_file(GRID_SHP_PATH)
    
    # 2. Get Years from LOYO files
    # Updated Filename format: loyo_base_score_annual_binary_2001.tif
    pred_files = sorted(list(PRED_DIR.glob("loyo_base_score_annual_binary_*.tif")))
    years = [int(f.stem.split('_')[-1]) for f in pred_files]
    print(f"Years found: {years}")

    # 3. Pre-load Basemap (Optional context)
    try:
        world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    except:
        world = None

    # 4. Process Loop
    for year in tqdm(years, desc="Processing Years"):
        
        # --- A. Load Prediction (Master Grid) ---
        # Updated filename to match base_score output
        pred_path = PRED_DIR / f"loyo_base_score_annual_binary_{year}.tif"
        
        with rio.open(pred_path) as src:
            pred_mask = src.read(1).astype(bool)
            profile = src.profile
            transform = src.transform
            crs = src.crs
            shape = (src.height, src.width)

        # Reproject Grid Shapefile and World to Raster CRS
        if gdf_grid.crs != crs:
            gdf_grid_proj = gdf_grid.to_crs(crs)
        else:
            gdf_grid_proj = gdf_grid

        if world is not None and world.crs != crs:
            world = world.to_crs(crs)

        # --- B. Create Pixel Area Map (Ha) ---
        area_grid_ha = get_pixel_area_hectares(shape, transform, crs)

        # --- C. Rasterize Grid IDs ---
        shapes = ((geom, val) for geom, val in zip(gdf_grid_proj.geometry, gdf_grid_proj[GRID_ID_COL]))
        grid_id_raster = rasterize(
            shapes=shapes,
            out_shape=shape,
            transform=transform,
            fill=-1, # Pixels outside shapefile
            dtype='int32'
        )
        
        unique_ids = gdf_grid_proj[GRID_ID_COL].unique()

        # --- D. Load Reference Data ---
        firecci_mask = get_annual_reference_mask(
            year, FIRECCI_DIR, 
            lambda y, m: f"cems_e5l_firecci_{y}_{m}_with_fraction.tif", 
            profile
        )
        
        mcd_mask = get_annual_reference_mask(
            year, MCD_DIR, 
            lambda y, m: f"cems_e5l_mcd_{y}_{m}_with_fraction.tif", 
            profile
        )

        # --- E. Summarize Zonal Stats ---
        stats_pred = summarize_burn_per_grid(pred_mask, area_grid_ha, grid_id_raster, unique_ids)
        stats_cci = summarize_burn_per_grid(firecci_mask, area_grid_ha, grid_id_raster, unique_ids)
        stats_mcd = summarize_burn_per_grid(mcd_mask, area_grid_ha, grid_id_raster, unique_ids)

        # --- F. Merge to GeoDataFrame for Plotting ---
        gdf_plot = gdf_grid_proj.copy()
        
        gdf_plot['ha_pred'] = gdf_plot[GRID_ID_COL].map(stats_pred).fillna(0)
        gdf_plot['ha_cci'] = gdf_plot[GRID_ID_COL].map(stats_cci).fillna(0)
        gdf_plot['ha_mcd'] = gdf_plot[GRID_ID_COL].map(stats_mcd).fillna(0)
        
        # --- G. Determine Common Scale (Max Hectares) ---
        vmax = max(gdf_plot['ha_pred'].max(), gdf_plot['ha_cci'].max(), gdf_plot['ha_mcd'].max())
        if vmax == 0: vmax = 1 

        # --- H. Plot ---
        fig, axes = plt.subplots(1, 3, figsize=(20, 8))
        fig.patch.set_facecolor('white') # Clean background
        
        # Updated plot title for clarity
        plot_configs = [
            (axes[0], 'ha_pred', f"Predicted (Base Score) {year}"),
            (axes[1], 'ha_cci', f"FireCCI {year}"),
            (axes[2], 'ha_mcd', f"MCD64A1 {year}")
        ]

        # Get bounds for consistent zooming
        minx, miny, maxx, maxy = gdf_plot.total_bounds

        for ax, col, title in plot_configs:
            # 1. Plot Basemap (if available)
            if world is not None:
                world.plot(ax=ax, color='#f0f0f0', edgecolor='white')
            
            # 2. Plot Grid outlines (subtle)
            gdf_plot.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=0.05, alpha=0.2)
            
            # 3. Plot Burned Area
            gdf_burned = gdf_plot[gdf_plot[col] > 0]
            
            if not gdf_burned.empty:
                gdf_burned.plot(
                    column=col, 
                    ax=ax, 
                    cmap='Reds', 
                    vmin=0, 
                    vmax=vmax,
                    legend=False
                )

            ax.set_title(title, fontsize=14, fontweight='bold')
            ax.set_xlim(minx, maxx)
            ax.set_ylim(miny, maxy)
            ax.axis('off')
        
        # Shared Colorbar
        cax = fig.add_axes([0.92, 0.25, 0.015, 0.5]) # [left, bottom, width, height]
        sm = plt.cm.ScalarMappable(cmap='Reds', norm=plt.Normalize(vmin=0, vmax=vmax))
        sm._A = []
        cbar = fig.colorbar(sm, cax=cax)
        cbar.set_label('Burned Area (Hectares)', fontsize=12)

        out_path = OUT_DIR / f"grid_burned_area_{year}.png"
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"Saved: {out_path.name} (Max Ha: {vmax:.1f})")

if __name__ == "__main__":
    main()

Processing Years: 100%|██████████| 22/22 [11:27<00:00, 31.25s/it]


In [12]:
't'

't'

Now save burned area per year

In [18]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Step 2 (Optimized): Calculate Burned Area (LOYO Base Score Model)
-- Reads PRE-COMPUTED Annual Binary Masks directly --
-- Summarizes Mha per Ecoregion --
"""

import os
import pandas as pd
import geopandas as gpd
import rasterio as rio
from rasterio.features import geometry_mask
import numpy as np
from pathlib import Path
from tqdm import tqdm

# ============================
# CONFIG
# ============================
YEARS  = list(range(2001, 2023)) 

# DIRECTORY POINTING TO YOUR PRE-MADE BINARY TIFFS
PRED_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/predictions_loyo_base_score/annual_binary_masks")

# Output Paths for LOYO base_score summary
OUT_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/burned_area_summaries_loyo_base_score")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "ba_ecoregion_loyo_base_score.csv"

# Ecoregion shapefile
ECOS_PATH = "/explore/nobackup/people/spotter5/helene/raw/merge_eco_v2.shp"
ECO_ID_COL = "ecoregion"

# ============================
# HELPERS
# ============================

def get_pixel_area_grid(shape, transform, crs):
    """
    Calculates the area of every pixel in the grid in Mega-Hectares (Mha).
    """
    height, width = shape
    res_x = abs(transform.a)
    res_y = abs(transform.e)

    if crs.is_geographic:
        # Latitude dependent area calculation for WGS84
        rows = np.arange(height) + 0.5
        _, lats = rio.transform.xy(transform, rows, np.zeros_like(rows), offset='center')
        lats = np.array(lats)
        lat_rads = np.radians(lats)
        
        # 1 degree latitude ~ 111,320 meters
        pixel_width_m = res_x * 111320 * np.cos(lat_rads)
        pixel_height_m = res_y * 111320
        
        # Area in m2 per row
        row_areas_m2 = pixel_width_m * pixel_height_m
        
        # Broadcast to full grid (Same lat = same area)
        area_grid_m2 = row_areas_m2[:, np.newaxis] * np.ones((1, width))
    else:
        # Projected CRS (meters)
        pixel_area_m2 = res_x * res_y
        area_grid_m2 = np.full(shape, pixel_area_m2)

    return area_grid_m2 / 1e10 

# ============================
# MAIN
# ============================

def main():
    if not PRED_DIR.exists():
        raise FileNotFoundError(f"Prediction directory not found: {PRED_DIR}")

    print(f"Loading ecoregions from: {ECOS_PATH}")
    ecos = gpd.read_file(ECOS_PATH)
    results = []

    print(f"Scanning years {min(YEARS)} to {max(YEARS)}...")
    
    for year in tqdm(YEARS, desc="Processing Years"):
        
        # 1. Load the Pre-Computed Annual Binary Mask
        tif_path = PRED_DIR / f"loyo_base_score_annual_binary_{year}.tif"
        
        if not tif_path.exists(): 
            continue
            
        with rio.open(tif_path) as src:
            # Read the binary data (0s and 1s) and convert to boolean mask
            annual_mask = src.read(1).astype(bool)
            transform = src.transform
            crs = src.crs

        # 2. Align Ecoregions to Raster CRS
        if ecos.crs != crs: 
            ecos_proj = ecos.to_crs(crs)
        else: 
            ecos_proj = ecos

        # 3. Create Area Grid (Mha)
        pixel_area_map = get_pixel_area_grid(annual_mask.shape, transform, crs)
        height, width = annual_mask.shape
        
        for idx, row in ecos_proj.iterrows():
            eco_id = row[ECO_ID_COL]
            geom = row.geometry
            
            if geom is None or geom.is_empty: continue
                
            try:
                eco_mask = geometry_mask(
                    [geom], 
                    transform=transform, 
                    invert=True, 
                    out_shape=(height, width), 
                    all_touched=False
                )
                
                # Intersect Burned Mask AND Ecoregion Mask
                burned_in_eco_mask = annual_mask & eco_mask
                
                ba_Mha = 0.0
                if burned_in_eco_mask.any():
                    ba_Mha = pixel_area_map[burned_in_eco_mask].sum()
                
                results.append({
                    "ecoregion": eco_id,
                    "year": year,
                    "ba_pred_loyo_base_score_Mha": ba_Mha # Matches the plot script column name
                })
                
            except Exception as e:
                print(f"Error {eco_id} {year}: {e}")

    # 4. Save Results
    if results:
        df_results = pd.DataFrame(results)
        df_results.to_csv(OUT_CSV, index=False)
        print(f"\n✅ DONE. Results saved to:\n{OUT_CSV}")
        
        # Quick print of total burned area
        total_ba = df_results["ba_pred_loyo_base_score_Mha"].sum()
        print(f"Total Burned Area (All Years/Regions): {total_ba:.4f} Mha")
    else:
        print("\n⚠️ No results generated.")

if __name__ == "__main__":
    main()

Processing Years: 100%|██████████| 22/22 [02:31<00:00,  6.88s/it]


Now extract burned area and compare to other products per ecoregion in multipanel way

Now make multipanel burned area comparison plot

Now extract burned area and compare to other products per ecoregion in multipanel way

In [21]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Compare LOYO Base Score Predictions to MCD64A1 and FireCCI native products (2001-2022).
-- LEAVE-ONE-YEAR-OUT VERSION --

Features:
- Plots LOYO predictions (using native base_score probabilities and fixed threshold).
- Compares against MCD64A1 and FireCCI.
- Includes individual ecoregion panels and a summed "Total" panel.
- Constrained to 2001-2022 based on reference data availability.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from pathlib import Path

# Set non-interactive backend for server environments
plt.switch_backend('Agg')

# ============================
# CONFIG
# ============================

# Directory containing the REFERENCE data (MCD/FireCCI)
REF_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/burned_area_summaries")

# Directory for NEW outputs (LOYO Base Score)
OUT_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/burned_area_summaries_loyo_base_score")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Files
BASE_CSV     = REF_DIR / "burned_area_by_ecoregion_predictions.csv"  # Contains MCD/FireCCI cols

# UPDATED: Pointing to the NEW 'LOYO Base Score' predictions CSV
NEW_PRED_CSV = OUT_DIR / "ba_ecoregion_loyo_base_score.csv" 
FINAL_CSV    = OUT_DIR / "burned_area_merged_loyo_base_score.csv"
OUT_PNG      = OUT_DIR / "burned_area_plot_loyo_base_score.png"

# Column Names
ECO_ID_COL  = "ecoregion"
MCD_COL     = "ba_mcd_native_Mha"
FIRECCI_COL = "ba_firecci_native_Mha"

# UPDATED: Column name from the LOYO CSV
PRED_COL    = "ba_pred_loyo_base_score_Mha"

# UPDATED: Stop exactly at 2022
YEAR_START, YEAR_END = 2001, 2022

# EXCLUSIONS
EXCLUDE_ECOS = {"WATER", "MIXED WOOD SHIELD", "TEMPERATE PRAIRIES", "WESTERN CORDILLERA"}

# COLORS
COLORS = {
    MCD_COL: "#2c3e50",      # Slate Grey
    FIRECCI_COL: "#e67e22",  # Vivid Orange
    PRED_COL: "#16a085"      # Deep Teal
}

def nice_pred_label(colname: str) -> str:
    if "loyo" in colname:
        return "LOYO Pred (Base Score)"
    return colname

# ============================
# MAIN
# ============================

def main():
    print(f"Loading Base Reference Data from: {BASE_CSV}")
    if not BASE_CSV.exists():
        raise FileNotFoundError(f"Base CSV not found: {BASE_CSV}")
        
    print(f"Loading New Predictions from: {NEW_PRED_CSV}")
    if not NEW_PRED_CSV.exists():
        raise FileNotFoundError(f"Prediction CSV not found: {NEW_PRED_CSV}")

    # --- 1. Load, Filter and Merge ---
    df_base = pd.read_csv(BASE_CSV)
    df_pred = pd.read_csv(NEW_PRED_CSV)
    
    # Merge on Ecoregion and Year
    df = df_base.merge(df_pred, on=[ECO_ID_COL, "year"], how="left")
    
    # Filter years (Caps strictly at 2022)
    df = df[(df["year"] >= YEAR_START) & (df["year"] <= YEAR_END)].copy()
    
    # Save merged data for inspection
    df.to_csv(FINAL_CSV, index=False)
    print(f"Merged CSV saved to: {FINAL_CSV}")

    # --- 2. Prepare Subplots ---
    ecos_all = sorted(df[ECO_ID_COL].dropna().unique())
    ecos_list = [e for e in ecos_all if e not in EXCLUDE_ECOS]
    
    # Let's keep your original order (Total last)
    plot_list = ecos_list + ["TOTAL BURNED AREA"]
    
    n_panels = len(plot_list)
    ncols = 4
    nrows = int(np.ceil(n_panels / ncols))

    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols, 
        figsize=(4 * ncols, 3.5 * nrows), 
        sharex=True
    )
    axes = axes.flatten()

    # --- 3. Plotting Loop ---
    # Create manual legend handles since we iterate
    legend_handles = [
        mlines.Line2D([], [], color=COLORS[MCD_COL], marker='o', markersize=4, label='MCD64A1'),
        mlines.Line2D([], [], color=COLORS[FIRECCI_COL], marker='s', markersize=4, label='Fire CCI'),
        mlines.Line2D([], [], color=COLORS[PRED_COL], marker='^', markersize=4, label=nice_pred_label(PRED_COL))
    ]

    for i, title in enumerate(plot_list):
        ax = axes[i]
        
        if title == "TOTAL BURNED AREA":
            # Filter out excluded ecos for the total calculation to be consistent
            df_valid = df[~df[ECO_ID_COL].isin(EXCLUDE_ECOS)]
            df_plot = df_valid.groupby("year")[[MCD_COL, FIRECCI_COL, PRED_COL]].sum().reset_index()
            
            ax.set_facecolor('#fdfefe') # Light highlight for total panel
            # Add a border to highlight Total
            for spine in ax.spines.values():
                spine.set_edgecolor('#333333')
                spine.set_linewidth(1.5)
        else:
            df_plot = df[df[ECO_ID_COL] == title].sort_values("year")

        # Plot datasets
        ax.plot(df_plot["year"], df_plot[MCD_COL], marker="o", markersize=4, 
                color=COLORS[MCD_COL], linewidth=1.5, alpha=0.8)
        
        ax.plot(df_plot["year"], df_plot[FIRECCI_COL], marker="s", markersize=4, 
                color=COLORS[FIRECCI_COL], linewidth=1.5, alpha=0.8)
        
        # Plot Predictions
        ax.plot(df_plot["year"], df_plot[PRED_COL], marker="^", markersize=4, 
                color=COLORS[PRED_COL], linewidth=2.0)

        ax.set_title(str(title), fontsize=10, fontweight='bold')
        ax.grid(True, ls=":", alpha=0.6)
        ax.tick_params(axis='both', labelsize=8)
        
        # Set X-ticks to perfectly align with 2001-2022 (Step by 3 years to prevent crowding)
        ax.set_xticks(np.arange(YEAR_START, YEAR_END + 1, 3))

        # Axis labeling
        if i >= (n_panels - ncols):
            ax.set_xlabel("Year", fontsize=10)
        if i % ncols == 0:
            ax.set_ylabel("Burned Area (Mha)", fontsize=10)

        # Handle scaling: Start at 0
        ax.set_ylim(bottom=0)

    # Clean up empty subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    # Global legend
    fig.legend(
        handles=legend_handles,
        loc="lower center", 
        ncol=3, 
        fontsize=12,
        frameon=False,
        bbox_to_anchor=(0.5, 0.0)
    )

    plt.tight_layout(rect=[0, 0.05, 1, 0.98]) # Make room for legend at bottom
    plt.savefig(OUT_PNG, dpi=250, bbox_inches="tight")
    plt.close()

    print(f"✅ Comparison plot saved to:\n   {OUT_PNG}")

if __name__ == "__main__":
    main()

In [15]:
't'

't'

Just to compare burned aarea